# 02 — Chunking, Embeddings, Vector DB & Retriever

This notebook covers milestone points **4–6**:
4. Chunking + embeddings.
5. Persistent vector database.
6. Retriever.

The objective is to convert the extracted technical documentation into searchable semantic knowledge.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import MANUAL_DIR, VECTOR_DIR, COLLECTION_NAME

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
print('Manuals:', MANUAL_DIR)
print('Vector DB:', VECTOR_DIR)


Manuals: C:\Users\se25479\Desktop\Factory_Floor_Chatbot\data\manuals
Vector DB: C:\Users\se25479\Desktop\Factory_Floor_Chatbot\data\vectorstore


## 1. Reload the PDFs

Each page keeps the filename, page number and equipment type so that retrieval results remain traceable.

In [2]:
from factory_floor.ingestion import load_manuals

documents = load_manuals(MANUAL_DIR)
print('Loaded pages:', len(documents))
if not documents:
    raise RuntimeError('No PDFs found. Run download_manuals.py or add PDFs to data/manuals/.')


Loaded pages: 3679


## 2. Chunk the documents

For the first baseline we use approximately **800 characters per chunk with 120 characters of overlap**. This is deliberately simple. Later we can benchmark it against a second chunking strategy instead of pretending the first choice is automatically optimal.

In [3]:
from factory_floor.vectorstore import chunk_documents

chunks = chunk_documents(documents)

print('Total chunks:', len(chunks))
print('Example metadata:', chunks[0].metadata)
print('Example text:', chunks[0].page_content[:700])


Total chunks: 12301
Example metadata: {'producer': 'Adobe PDF Library 11.0', 'creator': 'Acrobat PDFMaker 11 für Word', 'creationdate': '2017-01-18T10:17:27+01:00', 'author': 'Siemens AG, DF MC', 'comments': '', 'company': 'Siemens AG', 'keywords': 'A5E39910322B AA; 01/2017', 'moddate': '2017-01-26T08:33:02+01:00', 'sourcemodified': 'D:20170118091131', 'subject': 'Compact Operating Instructions', 'title': 'CU240B-2 and CU240E-2 Control Units', 'company-long': 'Siemens AG', 'company-short': 'Siemens', 'document-class': 'Compact Operating Instructions', 'document-class-mrl': '', 'edition': '01/2017', 'order-nr': 'A5E39910322B AA', 'print-year': '2015 - 2017', 'product-group': 'SINAMICS G120', 'system': 'SINAMICS', 'source': 'C:\\Users\\se25479\\Desktop\\Factory_Floor_Chatbot\\data\\manuals\\Siemens_CU240B2_CU240E2_Operating_Instructions.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Siemens_CU240B2_CU240E2_Operating_Instructions.pdf', 'manufacturer': 'Siemens', 'e

## 3. Create embeddings

An embedding converts each chunk into a numerical vector that represents semantic meaning. Similar meanings should be located near each other in vector space.

This milestone uses `text-embedding-3-small` because it is inexpensive and sufficient for a first baseline.

In [4]:
import os
from factory_floor.vectorstore import get_embeddings

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in a .env file before running this cell.')

embeddings = get_embeddings()
print('Embedding model ready.')


Embedding model ready.


## 4. Build the persistent Chroma vector database

The database is persisted under `data/vectorstore/`, so the Streamlit app does not need to re-embed every PDF each time it starts.

In [5]:
from factory_floor.vectorstore import build_vectorstore

# Rebuild from scratch for a clean milestone run.
vectorstore = build_vectorstore(
    chunks,
    persist_directory=VECTOR_DIR,
    collection_name=COLLECTION_NAME,
    embeddings=embeddings,
    rebuild=True,
)
print('Vector DB built at:', VECTOR_DIR)


Vector DB built at: C:\Users\se25479\Desktop\Factory_Floor_Chatbot\data\vectorstore


## 5. Create the retriever

For the baseline, retrieve the **top 5** semantically related chunks.

In [6]:
from factory_floor.rag import build_retriever

retriever = build_retriever(vectorstore, k=5)
print('Retriever ready.')


Retriever ready.


## 6. Test retrieval before adding any LLM

This test is intentionally independent from generation. If retrieval is poor, an LLM cannot reliably repair the missing evidence.

In [7]:
query = 'What should be checked when an electric motor is overheating or showing excessive vibration?'
results = retriever.invoke(query)

print('QUERY:', query)
for i, doc in enumerate(results, 1):
    page = doc.metadata.get('page', 0) + 1
    print(f'\n--- RESULT {i} | {doc.metadata.get("source_file")} | page {page} ---')
    print(doc.page_content[:900])

QUERY: What should be checked when an electric motor is overheating or showing excessive vibration?

--- RESULT 1 | Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf | page 101 ---
All the fixing bolts/screws for the mechanical and electrical connections have 
been securely tightened
  X
All the potential connections, grounding connections and shield supports are 
correctly seated and properly bonded
  X
The winding insulation resistances are sufficiently high   X
Any bearing insulation is fitted as shown on the plates and labels   X
The CABLES and insulating parts and components are in good condition and 
there is no evidence of discoloring
  X
(*) You can perform these checks while the motor is at standstill or, if required, while running.
NOTICE
Machine damage
When carrying out the inspection, if you detect any impermissible deviations from the normal 
state, you must rectify them immediately. They may otherwise cause damage to the machine.

--- RESULT 2 | Siemens_SINAMICS_G120C_O

## 7. Optional metadata-filtered retrieval

Because chunks carry `equipment_type`, the same vector database can later restrict a query to motors or VFDs when the agent already knows which equipment is involved.

In [8]:
from factory_floor.rag import build_retriever

motor_retriever = build_retriever(vectorstore, k=5, equipment_type='electric_motor')
vfd_retriever = build_retriever(vectorstore, k=5, equipment_type='VFD')

print('Motor and VFD filtered retrievers created.')


Motor and VFD filtered retrievers created.


## Milestone checkpoint

At this point we have a complete retrieval layer:

`PDFs → pages → chunks → embeddings → Chroma → retriever`

The next notebook adds the LLM and produces grounded answers with source citations.